# Scope, Closures, and the Names Python Remembers
This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Scope, Closures, and the Names Python Remembers** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Scope, Closures, and the Names Python Remembers**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Use LEGB as an explanatory model rather than a slogan
- See closures as remembered references
- Understand `nonlocal` without fear
- Connect scope to function factories and decorators


A closure keeps access to objects referenced from the enclosing scope. What is remembered is not “the text of the variable”, but the live binding relationship needed by the inner function.


In [1]:
def make_counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc

c = make_counter()
print(c(), c(), c())
print(c.__closure__)


1 2 3
(<cell at 0x000002A8BEF379A0: int object at 0x00007FFAF014D368>,)


Disassembly can reveal free variables and closure-related behavior. You do not need to become a compiler engineer; you only need to notice that nested functions carry extra runtime structure.


In [2]:
import dis

def outer():
    x = 10
    def inner():
        return x + 1
    return inner

dis.dis(outer)


              0 MAKE_CELL                1 (x)

  3           2 RESUME                   0

  4           4 LOAD_CONST               1 (10)
              6 STORE_DEREF              1 (x)

  5           8 LOAD_CLOSURE             1 (x)
             10 BUILD_TUPLE              1
             12 LOAD_CONST               2 (<code object inner at 0x000002A8BEFA06B0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_28496\2315492634.py", line 5>)
             14 MAKE_FUNCTION            8 (closure)
             16 STORE_FAST               0 (inner)

  7          18 LOAD_FAST                0 (inner)
             20 RETURN_VALUE

Disassembly of <code object inner at 0x000002A8BEFA06B0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_28496\2315492634.py", line 5>:
              0 COPY_FREE_VARS           1

  5           2 RESUME                   0

  6           4 LOAD_DEREF               0 (x)
              6 LOAD_CONST               1 (1)
              8 BINARY_OP                0 (+)


Python checks local, enclosing, global, then built-in names.

That is why inner functions can still use outer names later.

It says “when I assign, I mean the enclosing binding, not a fresh local one”.

They are not only theory. They help with stateful callbacks, factories, and decorators.


The name is found in the enclosing scope because it is not local to the inner function.


In [3]:
def outer():
    msg = "from outer"
    def inner():
        return msg
    return inner()

print(outer())


from outer


The returned function keeps the outer value around.


In [4]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor
    return multiply

double = make_multiplier(2)
print(double(10))


20


Without `nonlocal`, this rebinding would create a new local variable instead.


In [5]:
def make_total():
    total = 0
    def add(value):
        nonlocal total
        total += value
        return total
    return add

add = make_total()
print(add(5), add(7))


5 12


This is not only a classroom topic. **Scope, Closures, and the Names Python Remembers** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Memorizing LEGB letters without really understanding lookup
- Using globals when a closure would keep state cleaner
- Forgetting that `nonlocal` is about rebinding, not only reading


- What is a closure?
- What does `nonlocal` do?
- Why can a nested function still see an outer name after the outer function returns?


- Closures are remembered access to enclosing names.
- LEGB is a useful runtime search order.
- `nonlocal` controls rebinding of enclosing names.


If this notebook did its job, **Scope, Closures, and the Names Python Remembers** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
